In [1]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Running locally (not in Colab)")

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Set project paths
PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/BigData/' if IN_COLAB else './'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/models', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Colab Environment: {IN_COLAB}")

Mounted at /content/drive
Project Root: /content/drive/MyDrive/Colab_Projects/BigData/
Colab Environment: True


In [2]:
# Install required packages for Colab
import subprocess

packages_to_install = [
    'pyspark',
    'shap',
    'lime',
    'gradio',
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'plotly'
]

for package in packages_to_install:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("\nAll packages installed successfully!")

✓ pyspark already installed
✓ shap already installed
Installing lime...
✓ gradio already installed
✓ tensorflow already installed
Installing scikit-learn...
✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ plotly already installed

All packages installed successfully!


# Initialize PySpark with optimized Colab configuration
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, MultilayerPerceptronClassifier

# Spark Configuration for Colab
spark = SparkSession.builder \
    .appName("BigDataAnomalyDetection") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.rdd.compress", "true") \
    .config("spark.shuffle.compress", "true") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark App: {spark.sparkContext.appName}")
print(f"Spark Configuration:")
for key, value in spark.sparkContext.getConf().getAll():
    if 'memory' in key.lower() or 'partition' in key.lower():
        print(f"  {key}: {value}")

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ARCHITECTURE DOCUMENTATION
architecture_doc = """
=== DISTRIBUTED SYSTEM ARCHITECTURE ===

1. DATA PARTITIONING STRATEGY:
   - Primary: Temporal partitioning (by hour/day)
   - Secondary: Hash partitioning by source_ip (200 partitions)
   - Rationale: Enables efficient time-series queries and parallel processing of IP traffic

2. CLUSTER CONFIGURATION (Colab-Optimized):
   - Driver Memory: 8GB (max safe for Colab)
   - Shuffle Partitions: 200 (balanced for 1M records)
   - Default Parallelism: 200 (matches shuffle partitions)
   - Adaptive Query Execution: Enabled (auto-tunes partitions)

3. DATA VARIETY SPECIFICATION:
   - Structured: Network metadata (timestamp, IPs, ports, protocols)
   - Numerical: Packet counts, bytes transferred, duration
   - Categorical: Protocol types, flags, anomaly labels
   - Temporal: Time-based patterns and behaviors

4. SCALABILITY JUSTIFICATION:
   - Lazy evaluation prevents memory overflow
   - Partition-based processing distributes computation
   - No .collect() on full dataset (only aggregated results)
"""

print(architecture_doc)
with open(f'{PROJECT_ROOT}/ARCHITECTURE.md', 'w') as f:
    f.write(architecture_doc)

In [ ]:
# Generate 1.2M network traffic records
np.random.seed(42)

n_records = 1_200_000  # 1.2 Million records
anomaly_ratio = 0.05   # 5% anomalies

# Generate base timestamp
start_date = datetime(2024, 1, 1)
timestamps = [start_date + timedelta(seconds=int(i * 86400 / (n_records / 365))) for i in range(n_records)]

# Generate source and destination IPs
source_ips = [f"192.168.{np.random.randint(0, 255)}.{np.random.randint(0, 255)}" for _ in range(n_records)]
dest_ips = [f"10.0.{np.random.randint(0, 255)}.{np.random.randint(0, 255)}" for _ in range(n_records)]

# Generate network features
protocols = np.random.choice(['TCP', 'UDP', 'ICMP', 'DNS'], n_records, p=[0.5, 0.3, 0.1, 0.1])
ports = np.random.choice([22, 80, 443, 3306, 5432, 9200, np.random.randint(10000, 60000)], n_records)

# Generate numerical features with anomaly injection
packet_count = np.random.exponential(100, n_records).astype(int) + 10
bytes_transferred = (packet_count * np.random.exponential(500, n_records)).astype(int)
duration_ms = (bytes_transferred / np.random.uniform(100, 1000, n_records)).astype(int)

# Inject anomalies
anomaly_indices = np.random.choice(n_records, int(n_records * anomaly_ratio), replace=False)
labels = np.zeros(n_records, dtype=int)

for idx in anomaly_indices:
    labels[idx] = 1
    packet_count[idx] = np.random.randint(5000, 50000)  # Abnormally high
    bytes_transferred[idx] = np.random.randint(100_000_000, 500_000_000)  # DDoS-like

# Create DataFrame before Spark
data_dict = {
    'timestamp': timestamps,
    'source_ip': source_ips,
    'dest_ip': dest_ips,
    'protocol': protocols,
    'port': ports,
    'packet_count': packet_count,
    'bytes_transferred': bytes_transferred,
    'duration_ms': duration_ms,
    'is_anomaly': labels
}

pandas_df = pd.DataFrame(data_dict)

print(f"Generated Dataset Summary:")
print(f"  Total Records: {len(pandas_df):,}")
print(f"  Anomalies: {pandas_df['is_anomaly'].sum():,} ({pandas_df['is_anomaly'].mean()*100:.1f}%)")
print(f"  Date Range: {pandas_df['timestamp'].min()} to {pandas_df['timestamp'].max()}")
print(f"\nDataset Shape: {pandas_df.shape}")
print(f"Memory Usage: {pandas_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nFirst 5 rows:")
print(pandas_df.head())

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(pandas_df)

# Apply partitioning strategy
spark_df_partitioned = spark_df \
    .withColumn('hour', hour(col('timestamp'))) \
    .withColumn('day', dayofmonth(col('timestamp'))) \
    .repartition(200, col('source_ip'))  # Hash partitioning by source IP

# Cache for performance
spark_df_partitioned.cache()

# Store partition info
n_partitions = spark_df_partitioned.rdd.getNumPartitions()
partition_sizes = spark_df_partitioned.rdd.mapPartitions(lambda x: [len(list(x))]).collect()

print(f"=== SPARK DISTRIBUTED PARTITIONING ===")
print(f"Number of Partitions: {n_partitions}")
print(f"Average Records per Partition: {spark_df_partitioned.count() / n_partitions:.0f}")

# Use sorted() to avoid shadowing of min/max by pyspark.sql.functions
partition_sizes_sorted = sorted(partition_sizes)
print(f"Min Records in Partition: {partition_sizes_sorted[0]:,}")
print(f"Max Records in Partition: {partition_sizes_sorted[-1]:,}")
print(f"\nDataFrame Schema:")
spark_df_partitioned.printSchema()


In [ ]:
# Introduce realistic data quality issues
from pyspark.sql.functions import when

# Create dataset with missing values and outliers
df_dirty = spark_df_partitioned.withColumn(
    'packet_count',
    when(rand() < 0.02, None).otherwise(col('packet_count'))  # 2% missing
).withColumn(
    'bytes_transferred',
    when(rand() < 0.01, None).otherwise(col('bytes_transferred'))  # 1% missing
)

print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total Records: {df_dirty.count():,}")
print(f"\nNull Value Analysis:")
df_dirty.select([count(when(col(c).isNull(), c)).alias(c) for c in df_dirty.columns]).show()

In [ ]:
# ETL STEP 1: Handle Missing Values (Lazy Evaluation)
from pyspark.sql.functions import mean as spark_mean, col

df_cleaned = df_dirty

# Calculate mean for imputation (lazy)
mean_packet = df_dirty.select(spark_mean('packet_count')).collect()[0][0]
mean_bytes = df_dirty.select(spark_mean('bytes_transferred')).collect()[0][0]
mean_duration = df_dirty.select(spark_mean('duration_ms')).collect()[0][0]

# Impute with mean (lazy transformation)
df_cleaned = df_cleaned.fillna({
    'packet_count': int(mean_packet),
    'bytes_transferred': int(mean_bytes),
    'duration_ms': int(mean_duration)
})

print(f"\nAfter Imputation - Null Values:")
df_cleaned.select([count(when(col(c).isNull(), c)).alias(c) for c in df_cleaned.columns]).show()